In [1]:
import math
import sklearn
import numpy as np
import pandas as pd
import os
import csv
from tqdm import tqdm
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def same_seed(seed):
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [4]:
device="cuda" if torch.cuda.is_available()else'cpu'
config={
'seed':5201314,
'select_all':True,
'valid_ratio':0.2,
'n_epochs':100,
'batch_size':16,
'learning_rate':2e-4,
'early_stop':15,
'save_path':'./titan_model/titan_model.ckpt'
}

In [5]:
def train_valid_split(raw_train_data,valid_ratio, seed):
    valid_data_size=int(len(raw_train_data)*valid_ratio)
    train_data_size=len(raw_train_data)-valid_data_size
    train_subset,valid_subset=random_split(raw_train_data,lengths=[train_data_size,valid_data_size],generator=torch.Generator().manual_seed(seed))
    train_data=raw_train_data[train_subset.indices]
    valid_data=raw_train_data[valid_subset.indices]
    return train_data,valid_data

In [6]:
def select_feat(train_data,valid_data,test_data,select_all=True,k=3):
    y_train=train_data[:,0]
    y_valid=valid_data[:,0]
    x_train=train_data[:,1:]
    x_valid=valid_data[:,1:]
    x_test=test_data

    scaler=StandardScaler()
    x_train=scaler.fit_transform(x_train)
    x_valid=scaler.fit_transform(x_valid)
    x_test=scaler.transform(x_test)

    if select_all:
        feat_idx=list(range(x_train.shape[1]))
    else:
        correlations=[]
        for i in range(x_train.shape[1]):
            corr=np.corrcoef(x_train[:,i],y_train)[0,1]
            correlations.append(abs(corr))
        correlations=np.nan_to_num(correlations)
        feat_idx=np.argsort(correlations)[::-1][:k].tolist()
        print(f"[{k} Features selected by Correlation] indices: {feat_idx}")
    return x_train[:,feat_idx],x_valid[:,feat_idx],x_test[:,feat_idx],y_train,y_valid

In [7]:
class titan_dataset(Dataset):
    def __init__(self,features,targets=None):
        if targets is None:
            self.targets=targets
        else:
            self.targets=torch.FloatTensor(targets)
        self.features=torch.FloatTensor(features)

    def __getitem__(self, idx):
        if self.targets is None:
            return self.features[idx]
        else:
            return self.features[idx],self.targets[idx]

    def __len__(self):
        return len(self.features)

In [8]:
class titan_model(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.layers=nn.Sequential(
            nn.Linear(input_dim,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,16),
            nn.BatchNorm1d(16),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(16,1)
        )

    def forward(self,x):
        x=self.layers(x)
        x=x.squeeze(1)
        return x


In [9]:
def titan_trainer(train_loader,valid_loader,config,model,device):
    criterion=nn.BCEWithLogitsLoss()
    optimizer=torch.optim.AdamW(model.parameters(),lr=config['learning_rate'],weight_decay=1e-4)
    writer=SummaryWriter()
    if not os.path.isdir('./titan_model'):
        os.mkdir('./titan_model')
    n_epochs=config['n_epochs']
    best_loss=math.inf
    step=0
    early_stop_count=0

    for epoch in range(n_epochs):
        model.train()
        loss_record=[]
        train_pbar=tqdm(train_loader,position=0,leave=False)
        for x,y in train_pbar:
            x,y=x.to(device),y.to(device).float()
            optimizer.zero_grad()
            pred=model(x)
            loss=criterion(pred,y)
            loss.backward()
            optimizer.step()
            step+=1
            loss_record.append(loss.detach().item())
            train_pbar.set_description(f'Epoch{epoch+1}/{n_epochs}')
            train_pbar.set_postfix({'loss':loss.detach().item()})
        mean_train_loss=sum(loss_record)/len(loss_record)

        model.eval()
        loss_record=[]
        acc_record=[]
        for x,y in valid_loader:
            x,y=x.to(device),y.to(device).float()
            with torch.no_grad():
                pred=model(x)
                loss=criterion(pred,y)
                pred_a=torch.sigmoid(pred)
                pred_b=torch.round(pred_a)
                acc=(pred_b==y).float().mean()
            loss_record.append(loss.item())
            acc_record.append(acc.item())
        mean_valid_loss=sum(loss_record)/len(loss_record)
        mean_valid_acc=sum(acc_record)/len(acc_record)
        print(f'Epoch[{epoch+1}/{n_epochs}]: Train loss:{mean_train_loss:.4f},Valid loss:{mean_valid_loss:.4f},Valid acc:{mean_valid_acc}')

        if mean_valid_loss<best_loss:
            best_loss=mean_valid_loss
            torch.save(model.state_dict(),config['save_path'])
            print('saving model with loss{:.3f}'.format(best_loss))
            early_stop_count=0
        else:
            early_stop_count+=1

        if early_stop_count>=config['early_stop']:
            print('\nmodel is not improving, so we halt the training session.')
            return

In [10]:
train_data=pd.read_csv('/content/drive/MyDrive/titan_train.csv').values
test_data=pd.read_csv('/content/drive/MyDrive/titan_test.csv').values
train_data,valid_data=train_valid_split(train_data,config['valid_ratio'],config['seed'])
print(f'train_data_size:{train_data.shape},valid_data_size:{valid_data.shape},test_data_size:{test_data.shape}')
x_train,x_valid,x_test,y_train,y_valid=select_feat(train_data,valid_data,test_data)
print(f'the number of features:{x_train.shape[1]}')
train_dataset=titan_dataset(x_train,y_train)
valid_dataset=titan_dataset(x_valid,y_valid)
test_dataset=titan_dataset(x_test)
train_loader=DataLoader(train_dataset,batch_size=config['batch_size'],shuffle=True,pin_memory=True)
valid_loader=DataLoader(valid_dataset,batch_size=config['batch_size'],shuffle=True,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=config['batch_size'],shuffle=False,pin_memory=True)

train_data_size:(712, 8),valid_data_size:(177, 8),test_data_size:(418, 7)
the number of features:7


In [11]:
model=titan_model(input_dim=x_train.shape[1]).to(device)
titan_trainer(train_loader,valid_loader,config,model,device)

Epoch[1/100]: Train loss:0.7427,Valid loss:0.6856,Valid acc:0.546875
saving model with loss0.686


Epoch[2/100]: Train loss:0.6793,Valid loss:0.6306,Valid acc:0.7552083333333334
saving model with loss0.631


Epoch[3/100]: Train loss:0.6438,Valid loss:0.6061,Valid acc:0.7604166666666666
saving model with loss0.606


Epoch[4/100]: Train loss:0.6146,Valid loss:0.5882,Valid acc:0.78125
saving model with loss0.588


Epoch[5/100]: Train loss:0.5971,Valid loss:0.5806,Valid acc:0.8072916666666666
saving model with loss0.581


...

Epoch[96/100]: Train loss:0.4252,Valid loss:0.5456,Valid acc:0.7760416666666666


Epoch[97/100]: Train loss:0.4556,Valid loss:0.4120,Valid acc:0.8489583333333334


Epoch[98/100]: Train loss:0.4508,Valid loss:0.4287,Valid acc:0.859375


Epoch[99/100]: Train loss:0.4416,Valid loss:0.4311,Valid acc:0.859375


Epoch[100/100]: Train loss:0.4523,Valid loss:0.5582,Valid acc:0.7708333333333334


In [12]:
def predict(test_loader,model,device):
    model.eval()
    preds=[]
    for x in tqdm(test_loader):
        x=x.to(device)
        with torch.no_grad():
            pred=model(x)
            pred=torch.round(torch.sigmoid(pred))
            preds.append(pred.detach().cpu())
    preds=torch.cat(preds,dim=0).numpy().flatten()
    return preds.astype(int)

def save_pred(preds,file):
    with open(file,'w') as fp:
        writer=csv.writer(fp)
        writer.writerow(['PassengerId','Survived'])
        for i,p in enumerate(preds,start=892):
            writer.writerow([i,p])

model=titan_model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config['save_path']))
preds=predict(test_loader,model,device)
save_pred(preds,'titan_pred.csv')


100%|██████████| 27/27 [00:00<00:00, 1024.10it/s]
